# ResNet

## 0. 基本认识 ResNet Overview

ResNet 是微软研究院提出的深度残差网络，对应论文是 *Deep Residual Learning for Image Recognition*。它在 2015 年 ILSVRC 图像分类任务中表现很突出，也是后续很多视觉模型的基础 backbone。

它关注的问题是：当 CNN 继续加深时，为什么训练效果反而可能变差，以及怎样让很深的网络更容易训练。

核心背景可以概括为：

- VGG 证明了加深网络通常有帮助，但网络越深，训练难度和计算量都会增加。
- 普通深层网络会出现 degradation problem：层数增加后，训练误差反而上升，这不只是过拟合问题。
- ResNet 用残差连接 residual connection 让网络学习残差函数，而不是直接学习完整映射。
- 常见 ResNet 包括 ResNet-18、ResNet-34、ResNet-50、ResNet-101 和 ResNet-152。

简单理解：ResNet 不是让每一层都必须学出一个全新的复杂变换，而是允许网络在需要时直接保留原输入，只学习“需要修改的部分”。

---

## 1. 残差学习 Residual Learning

### 1.1 基本思想

普通网络希望若干层直接学习目标映射：

$$
H(x)
$$

ResNet 把这个目标改写成：

$$
H(x) = F(x) + x
$$

其中：

- $x$：输入特征。
- $H(x)$：希望得到的目标输出。
- $F(x)$：残差函数，也就是网络真正要学习的部分。
- $F(x) + x$：把残差分支和 shortcut 分支相加后的输出。

等价地，残差分支学习的是：

$$
F(x) = H(x) - x
$$

如果某些层暂时学不到有用变换，残差分支可以接近 0，这样输出就接近输入 $x$，网络至少不会因为堆得更深而明显退化。

### 1.2 Shortcut Connection

ResNet 中的 shortcut connection 通常有两种形式：

| 类型 | 写法 | 使用场景 |
| --- | --- | --- |
| Identity shortcut | 直接把 $x$ 加到输出上 | 输入和输出形状完全一致 |
| Projection shortcut | 用 $1 \times 1$ 卷积调整 $x$ | 通道数或空间尺寸不一致 |

最理想的情况是直接相加：

```text
out = F(x) + x
```

但如果残差分支输出是 $(B, 128, 28, 28)$，而输入 $x$ 是 $(B, 64, 56, 56)$，就不能直接相加，需要用 $1 \times 1$ 卷积和 stride=2 把 shortcut 分支也变成 $(B, 128, 28, 28)$。

### 1.3 为什么能缓解深层训练问题

残差连接主要有两个作用：

- 让梯度可以沿着 shortcut 更直接地向前层传播，缓解深层网络训练困难。
- 让网络更容易学习恒等映射 identity mapping，深层网络不必强迫每一层都改变特征。

所以 ResNet 的重点不是单纯“层数更多”，而是通过残差结构让更多层变得可训练。

---

## 2. 残差块 Residual Block

### 2.1 BasicBlock

BasicBlock 主要用于 ResNet-18 和 ResNet-34。一个 BasicBlock 通常包含两个 $3 \times 3$ 卷积：

```text
x -> 3x3 Conv -> BN -> ReLU -> 3x3 Conv -> BN -> + shortcut -> ReLU
```

写成公式可以理解为：

$$
y = ReLU(F(x) + shortcut(x))
$$

其中残差分支 $F(x)$ 由两个卷积层组成，shortcut 分支一般直接使用输入 $x$。

BasicBlock 的特点：

- 结构简单，主要堆叠 $3 \times 3$ 卷积。
- expansion 通常为 1，输出通道数就是 block 设定的通道数。
- 适合较浅的 ResNet，例如 18 层和 34 层。

### 2.2 Bottleneck

Bottleneck 主要用于 ResNet-50、ResNet-101 和 ResNet-152。它通常由三个卷积组成：

```text
x -> 1x1 Conv -> BN -> ReLU -> 3x3 Conv -> BN -> ReLU -> 1x1 Conv -> BN -> + shortcut -> ReLU
```

三个卷积的作用分别是：

| 卷积 | 作用 |
| --- | --- |
| $1 \times 1$ Conv | 降低通道数，减少计算量 |
| $3 \times 3$ Conv | 提取空间特征 |
| $1 \times 1$ Conv | 恢复或扩展通道数 |

Bottleneck 的 expansion 通常为 4。例如中间通道数是 64，block 最终输出通道数通常是 $64 \times 4 = 256$。

### 2.3 BasicBlock 和 Bottleneck 对比

| 对比项 | BasicBlock | Bottleneck |
| --- | --- | --- |
| 常用模型 | ResNet-18 / ResNet-34 | ResNet-50 / ResNet-101 / ResNet-152 |
| 卷积结构 | $3 \times 3 + 3 \times 3$ | $1 \times 1 + 3 \times 3 + 1 \times 1$ |
| expansion | 1 | 4 |
| 优点 | 简单直接 | 更适合构建很深的网络 |
| 主要目的 | 保持结构清晰 | 控制计算量和参数量 |

学习 ResNet 时，先把 BasicBlock 的加法关系弄清楚，再看 Bottleneck 会更容易。

---

## 3. 网络结构 Network Architecture

### 3.1 主线结构

以 ImageNet 版本 ResNet 为例，整体结构可以概括为：

```text
Input
-> Conv7x7, stride=2
-> MaxPool3x3, stride=2
-> conv2_x
-> conv3_x
-> conv4_x
-> conv5_x
-> Global Average Pooling
-> Fully Connected
```

其中 `conv2_x` 到 `conv5_x` 是由多个 residual block 堆叠出来的四个阶段。每进入一个新阶段，通常会让空间尺寸减半、通道数增加。

### 3.2 ResNet-18 结构

ResNet-18 使用 BasicBlock，每个阶段有 2 个 block：

| 阶段 | 主要结构 | 输出尺寸 |
| --- | --- | --- |
| Input | RGB 图像 | $3 \times 224 \times 224$ |
| Conv1 | $7 \times 7$ Conv, stride=2, 64 channels | $64 \times 112 \times 112$ |
| MaxPool | $3 \times 3$ MaxPool, stride=2 | $64 \times 56 \times 56$ |
| conv2_x | 2 个 BasicBlock, 64 channels | $64 \times 56 \times 56$ |
| conv3_x | 2 个 BasicBlock, 128 channels | $128 \times 28 \times 28$ |
| conv4_x | 2 个 BasicBlock, 256 channels | $256 \times 14 \times 14$ |
| conv5_x | 2 个 BasicBlock, 512 channels | $512 \times 7 \times 7$ |
| AvgPool | 全局平均池化 | $512 \times 1 \times 1$ |
| FC | 全连接分类层 | $1000$ |

### 3.3 不同 ResNet 配置

常见 ResNet 的区别主要在 block 类型和每个阶段的 block 数量：

| 模型 | Block 类型 | 每阶段 block 数 | 常见输出通道 |
| --- | --- | --- | --- |
| ResNet-18 | BasicBlock | 2, 2, 2, 2 | 64, 128, 256, 512 |
| ResNet-34 | BasicBlock | 3, 4, 6, 3 | 64, 128, 256, 512 |
| ResNet-50 | Bottleneck | 3, 4, 6, 3 | 256, 512, 1024, 2048 |
| ResNet-101 | Bottleneck | 3, 4, 23, 3 | 256, 512, 1024, 2048 |
| ResNet-152 | Bottleneck | 3, 8, 36, 3 | 256, 512, 1024, 2048 |

ResNet-34 和 ResNet-50 的每阶段 block 数相同，但 ResNet-50 使用 Bottleneck，所以层数更多，最终通道数也更大。

---

## 4. 尺寸和参数计算 Parameter Calculation

### 4.1 卷积输出尺寸

卷积或池化后的空间尺寸仍然使用通用公式：

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核或池化核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

ResNet 中常见的 $3 \times 3$ 卷积如果使用 `padding=1, stride=1`，空间尺寸保持不变：

$$
O = \frac{I + 2 \times 1 - 3}{1} + 1 = I
$$

如果使用 `stride=2`，空间尺寸大约减半。

### 4.2 下采样时的尺寸变化

以 ResNet-18 的阶段变化为例：

```text
224 -> Conv7x7 stride=2 -> 112
112 -> MaxPool stride=2 -> 56
56  -> conv2_x -> 56
56  -> conv3_x stride=2 -> 28
28  -> conv4_x stride=2 -> 14
14  -> conv5_x stride=2 -> 7
7   -> Global Average Pooling -> 1
```

每个新阶段的第一个 block 通常负责下采样，后面的 block 保持尺寸不变。

### 4.3 Shortcut 分支的形状匹配

残差相加要求两个张量形状完全一致：

```text
F(x).shape == shortcut(x).shape
```

如果输入和输出形状一样，可以直接 identity shortcut：

```text
shortcut(x) = x
```

如果空间尺寸或通道数变化，就需要 projection shortcut：

```text
shortcut(x) = Conv1x1(x, stride=2)
```

例如从 $64 \times 56 \times 56$ 变成 $128 \times 28 \times 28$ 时，残差分支第一个 $3 \times 3$ 卷积使用 stride=2，shortcut 分支也要用 $1 \times 1$ 卷积和 stride=2。否则两个分支不能相加。

### 4.4 参数量计算

卷积层参数量为：

$$
K_h \times K_w \times C_{in} \times C_{out} + C_{out}
$$

其中：

- $K_h, K_w$：卷积核高和宽。
- $C_{in}$：输入通道数。
- $C_{out}$：输出通道数。
- $C_{out}$：bias 参数数量，如果卷积后接 BatchNorm，常见实现会设 `bias=False`。

以 BasicBlock 中一个 $3 \times 3$ 卷积为例，如果输入输出都是 64 通道，不使用 bias：

$$
3 \times 3 \times 64 \times 64 = 36864
$$

如果是用于 downsample 的 $1 \times 1$ 卷积，从 64 通道变成 128 通道，不使用 bias：

$$
1 \times 1 \times 64 \times 128 = 8192
$$

ResNet 的参数量不只来自卷积主干，也包括 BatchNorm 参数和最后的全连接层。

---

## 5. 关键思想 Key Ideas

### 5.1 解决 degradation problem

ResNet 主要针对的是 degradation problem：当网络层数加深后，训练误差可能变高，说明更深的网络没有被成功优化好。

这和过拟合不同：

| 问题 | 表现 | 原因侧重点 |
| --- | --- | --- |
| 过拟合 Overfitting | 训练集好，验证集差 | 泛化能力不足 |
| 退化 Degradation | 训练集也变差 | 深层网络优化困难 |

残差连接让深层网络更容易学到“不改变输入”的恒等映射，因此缓解了层数增加带来的训练困难。

### 5.2 Identity Mapping

如果最优情况是不改变某些特征，普通网络需要一串卷积层主动学出恒等映射；ResNet 通过 shortcut 直接提供了这个路径。

直观理解：

```text
普通网络：每层都必须重新变换特征
ResNet：可以保留原特征，只学习额外补充或修正
```

这让网络深度增加时，至少可以退回到接近浅层网络的效果，而不是越堆越难训练。

### 5.3 BatchNorm 和 ReLU 的配合

经典 ResNet block 中常见顺序是：

```text
Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm -> Add -> ReLU
```

BatchNorm 可以让训练更稳定，ReLU 提供非线性。需要注意的是，残差分支最后一次卷积后通常先不立即 ReLU，而是先和 shortcut 相加，再统一 ReLU。

后续还有 pre-activation ResNet，把 BN 和 ReLU 放到卷积前面，但初学时先掌握经典 post-activation 结构即可。

### 5.4 和 VGG、GoogLeNet 的区别

| 对比项 | VGG | GoogLeNet | ResNet |
| --- | --- | --- | --- |
| 主要思路 | 规则堆叠小卷积核 | 多分支提取多尺度特征 | 残差连接训练更深网络 |
| 模块结构 | VGG block | Inception module | Residual block |
| 分类前结构 | 大型全连接层 | 全局平均池化 | 全局平均池化 |
| 关键难点 | 参数量大 | 分支拼接和通道配置 | 残差相加和形状匹配 |

ResNet 的结构看起来比 GoogLeNet 更直接，但 shortcut 的形状匹配非常关键。

---

## 6. PyTorch 实现思路 PyTorch Implementation

当前 `9.ResNet` 目录里只有 notebook，没有配套 `model.py`、`train.py` 或 `test.py`。所以下面按常见 PyTorch 写法整理，不对应某个本地源码文件。

### 6.1 BasicBlock 写法

BasicBlock 的核心是残差分支和 shortcut 分支相加：

```python
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)
        return out
```

这里的 `downsample` 通常是 $1 \times 1$ 卷积加 BatchNorm，用来让 shortcut 分支和残差分支形状一致。

### 6.2 Bottleneck 写法

Bottleneck 的结构是先降维、再做 $3 \times 3$ 卷积、最后升维：

```text
1x1 Conv -> BN -> ReLU
3x3 Conv -> BN -> ReLU
1x1 Conv -> BN
Add shortcut -> ReLU
```

Bottleneck 的常见写法如下，注意第三层卷积输出通道数是 `planes * expansion`，shortcut 分支也要匹配：

```python
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, planes, stride=1, downsample=None):
        super().__init__()
        width = planes
        self.conv1 = nn.Conv2d(in_channels, width, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, planes * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)
        return out
```

其中 `planes` 是中间层通道数，最终输出通道数是 `planes * expansion = planes * 4`。因此写 Bottleneck 时要特别注意最后一层卷积和 shortcut 分支的输出通道数。

### 6.3 构建 ResNet 主干

常见写法会定义 `_make_layer` 来批量堆叠 residual block：

```text
layer1 = _make_layer(block, 64,  blocks=2, stride=1)
layer2 = _make_layer(block, 128, blocks=2, stride=2)
layer3 = _make_layer(block, 256, blocks=2, stride=2)
layer4 = _make_layer(block, 512, blocks=2, stride=2)
```

对于 ResNet-18，`block=BasicBlock`，`blocks=[2, 2, 2, 2]`。

模型前向传播可以概括为：

```text
conv1 -> bn1 -> relu -> maxpool
-> layer1 -> layer2 -> layer3 -> layer4
-> avgpool -> flatten -> fc
```

### 6.4 小数据集实战调整

如果把 ResNet 用到 CIFAR-10、FashionMNIST 这类小数据集，常见需要调整：

- 输入通道：灰度图要把第一层 `in_channels` 从 3 改成 1。
- 输出类别：最后一层 `Linear` 的输出改成数据集类别数。
- 输入尺寸：如果不 resize 到 $224 \times 224$，第一层大卷积和早期 maxpool 可能过度下采样。
- 保存和加载路径：训练脚本和测试脚本要保持一致。

训练流程和前面的 CNN 类似：准备数据、构建模型、使用 `CrossEntropyLoss`、选择优化器、统计 loss 和 accuracy、保存验证集表现最好的参数。模型最后一层输出 logits，不需要手动加 Softmax。

---

## 7. 注意点 Common Pitfalls

### 7.1 残差相加前形状必须一致

残差分支和 shortcut 分支相加时，batch、通道数、高度和宽度都必须一致。常见错误是只让残差分支 stride=2，却忘了 shortcut 分支也要 downsample。

正确思路是：

```text
如果 out.shape != identity.shape，就用 downsample 调整 identity
```

否则会出现 tensor size mismatch。

### 7.2 不要把 BasicBlock 和 Bottleneck 的 expansion 写混

BasicBlock 的 expansion 一般是 1，Bottleneck 的 expansion 一般是 4。这个值会影响：

- 当前 block 的最终输出通道数。
- 下一层 block 的输入通道数。
- shortcut 分支是否需要 $1 \times 1$ 卷积。
- 最后全连接层的输入维度。

例如 ResNet-18 最后通常是 512 维特征，ResNet-50 最后通常是 2048 维特征。

### 7.3 下采样位置要和结构表对应

ResNet 通常在每个新阶段的第一个 block 做下采样，也就是 `layer2`、`layer3`、`layer4` 的第一个 block 使用 stride=2。后面的 block stride=1。

如果每个 block 都 stride=2，特征图会缩得太快；如果都不 stride=2，后面的全局池化前尺寸会和预期不一致。

### 7.4 分类任务不要在模型末尾手动加 Softmax

使用 `nn.CrossEntropyLoss()` 时，模型输出应该是原始 logits。`CrossEntropyLoss` 内部会处理 softmax 相关计算，所以最后保留 `Linear` 即可。

测试时如果要得到预测类别，可以直接使用：

```python
pred = torch.argmax(output, dim=1)
```

### 7.5 论文原版和实战改写版不要混写

| 对比项 | ImageNet 原版 ResNet | 小数据集实战版 |
| --- | --- | --- |
| 输入通道 | 3 通道 RGB | 1 通道或 3 通道 |
| 输入尺寸 | 常见 $224 \times 224$ | 可能是 $32 \times 32$、$28 \times 28$ 或 resize 后尺寸 |
| 第一层 | $7 \times 7$ Conv, stride=2 + MaxPool | 可能改成 $3 \times 3$ Conv，并去掉早期 MaxPool |
| 输出类别 | 1000 类 | 按数据集类别数设置 |
| 主要关注 | 大规模图像分类 | 结构理解和训练流程 |

画结构图、算尺寸或写代码时，要先确认自己采用的是论文原版还是为了当前数据集改过的版本。

---

## 8. 总结 Summary

- ResNet 的核心思想是残差学习，把目标映射写成 $H(x)=F(x)+x$。
- 残差连接让网络更容易学习恒等映射，也让梯度更容易传到前面的层。
- BasicBlock 常用于 ResNet-18 和 ResNet-34，结构是两个 $3 \times 3$ 卷积。
- Bottleneck 常用于 ResNet-50 及更深模型，结构是 $1 \times 1$、$3 \times 3$、$1 \times 1$ 三个卷积。
- ResNet 主干通常由 Conv1、MaxPool、conv2_x 到 conv5_x、全局平均池化和全连接分类层组成。
- 残差相加前必须保证两个分支的 shape 一致；通道数或尺寸变化时需要 downsample shortcut。
- 按论文结构复现时最容易错的是 stride 位置、downsample 分支、expansion 值、最后全连接输入维度，以及原版和小数据集改写版混用。

***********